# 08 — Multi-Run Regeneration (DKT + Code-DKT)

Regenera **DKT** e **Code-DKT** com **10 runs (seeds 42–51) × 5 assignments** para alinhar parcialmente com o protocolo do Shi et al. (2022) — que reporta `mean ± std` sobre 10 runs.

**Decisão metodológica:** BKT mantém-se em **1 run** (importado do `bkt_results.pkl` original) por dois motivos:
1. **Custo prohibitivo:** pyBKT (EM sequencial via fallback dos patches do `.venv`) leva ~45s por fit. 50 fits = 38min só para o BKT, sem prints intermediários do `nbconvert`. Tentativa anterior estourou timeout de 40min sem salvar nada.
2. **Determinismo esperado:** pyBKT é tipicamente determinístico em datasets pequenos como o CSEDM (EM converge ao mesmo ótimo independentemente da seed). 10 runs degenerariam em σ=0.

**Implicação para o 07_comparison:** BKT entra no Wilcoxon como ponto fixo (variância zero), e DKT vs Code-DKT é comparado com N=50 pares pareados (5 assignments × 10 seeds). Documentar como limitação metodológica.

**Outputs:**
- `results/bkt_results_multirun.pkl` — re-embala `bkt_results.pkl` em schema multirun (1 run)
- `results/dkt_results_multirun.pkl` — 10 runs × 5 assignments
- `results/code_dkt_results_multirun.pkl` — 10 runs × 5 assignments

---

## 1 — Setup

In [1]:
import os
import sys
import pickle
import random
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch

ROOT = Path(".").resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.models.dkt import train_and_evaluate as dkt_train_and_evaluate
from src.models.dkt import train_dkt, predict_dkt
from src.models.code_dkt import train_and_evaluate as cdkt_train_and_evaluate
from src.code_features import build_vocab
from src.evaluation import build_problem_index

print(f"Python {sys.version.split()[0]} | PyTorch {torch.__version__}")

# Reprodutibilidade
SEEDS = list(range(42, 52))   # 42..51 — 10 runs
SEED_DEFAULT = 42

def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_global_seed(SEED_DEFAULT)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

DATA_DIR = ROOT / "data" / "CSEDM"
RESULTS_DIR = ROOT / "results"


Python 3.12.3 | PyTorch 2.11.0+cu130
Device: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


In [2]:
with open(RESULTS_DIR / "sequences_bkt_dkt.pkl", "rb") as f:
    seqs = pickle.load(f)
ASSIGNMENT_IDS = seqs["assignment_ids"]
print(f"Assignments: {ASSIGNMENT_IDS}")

CACHE_PATH = RESULTS_DIR / "code_features_cache.pkl"
t0 = time.time()
with open(CACHE_PATH, "rb") as f:
    cache_raw = pickle.load(f)
print(f"Cache: {len(cache_raw):,} CodeStateIDs em {time.time()-t0:.1f}s")


Assignments: [439, 487, 492, 494, 502]


Cache: 53,990 CodeStateIDs em 0.7s


---

## 2 — Best configs (do grid search original)

In [3]:
with open(RESULTS_DIR / "dkt_results.pkl", "rb") as f:
    dkt_single = pickle.load(f)
BEST_DKT_CONFIG = dkt_single[ASSIGNMENT_IDS[0]]["config"]
print(f"DKT best config: {BEST_DKT_CONFIG}")

with open(RESULTS_DIR / "code_dkt_results.pkl", "rb") as f:
    cdkt_single = pickle.load(f)
BEST_CDKT_CONFIG = cdkt_single[ASSIGNMENT_IDS[0]]["config"]
print(f"Code-DKT best config: {BEST_CDKT_CONFIG}")

with open(RESULTS_DIR / "bkt_results.pkl", "rb") as f:
    bkt_single = pickle.load(f)
print(f"BKT single-run carregado: {list(bkt_single.keys())}")


DKT best config: {'hidden_dim': 200, 'dropout': 0.2, 'lr': 0.0005, 'batch_size': 128, 'epochs': 40, 'max_len': 50}


Code-DKT best config: {'hidden_dim': 200, 'dropout': 0.1, 'lr': 0.0005, 'batch_size': 128, 'epochs': 40, 'max_len': 50, 'R': 50}
BKT single-run carregado: [439, 487, 492, 494, 502]


---

## Helpers

In [4]:
def aggregate_runs(runs: list[dict]) -> dict:
    """Calcula mean/std das AUCs sobre N runs."""
    all_aucs   = np.array([r["all_auc"]   for r in runs], dtype=float)
    first_aucs = np.array([r["first_auc"] for r in runs], dtype=float)
    return {
        "all_auc_mean":   float(np.nanmean(all_aucs)),
        "all_auc_std":    float(np.nanstd(all_aucs, ddof=1)) if len(all_aucs)  > 1 else 0.0,
        "first_auc_mean": float(np.nanmean(first_aucs)),
        "first_auc_std":  float(np.nanstd(first_aucs, ddof=1)) if len(first_aucs) > 1 else 0.0,
    }

def fmt_pct(x): return f"{x*100:.2f}%"

def silent_train_and_evaluate_dkt(*args, **kwargs):
    """Wrapper que suprime os prints per-epoch do train_dkt."""
    import io
    from contextlib import redirect_stdout
    buf = io.StringIO()
    with redirect_stdout(buf):
        res = dkt_train_and_evaluate(*args, **kwargs)
    return res

def silent_train_and_evaluate_cdkt(*args, **kwargs):
    """Wrapper que suprime os prints per-epoch do train_code_dkt."""
    import io
    from contextlib import redirect_stdout
    buf = io.StringIO()
    with redirect_stdout(buf):
        res = cdkt_train_and_evaluate(*args, **kwargs)
    return res


---

## 3 — BKT: re-embala `bkt_results.pkl` em schema multirun (1 run)

Não treinamos novos BKTs (justificativa no header). Apenas adaptamos o pickle existente ao schema unificado com `'runs'` de tamanho 1, para que o `07_comparison.ipynb` consuma uniformemente.

In [5]:
bkt_multirun = {}
for aid in ASSIGNMENT_IDS:
    s = bkt_single[aid]
    auc_a = float(s["all_auc"]) if s["all_auc"] is not None else float("nan")
    auc_f = float(s["first_auc"]) if s["first_auc"] is not None else float("nan")
    runs = [{
        "seed":      SEED_DEFAULT,
        "all_auc":   auc_a,
        "first_auc": auc_f,
        "pred_df":   None,   # bkt_results.pkl não tem pred_df serializado
    }]
    bkt_multirun[aid] = {
        **aggregate_runs(runs),
        "runs":    runs,
        "n_train": s["n_train"],
        "n_test":  s["n_test"],
        "params":  s["params"],
    }

print("BKT re-embalado em schema multirun (1 run):")
for aid in ASSIGNMENT_IDS:
    m = bkt_multirun[aid]
    print(f"  A{aid}: first={fmt_pct(m['first_auc_mean'])} (n=1), all={fmt_pct(m['all_auc_mean'])} (n=1)")


BKT re-embalado em schema multirun (1 run):
  A439: first=63.21% (n=1), all=64.23% (n=1)
  A487: first=68.40% (n=1), all=69.07% (n=1)
  A492: first=54.20% (n=1), all=63.62% (n=1)
  A494: first=57.81% (n=1), all=59.66% (n=1)
  A502: first=56.92% (n=1), all=57.37% (n=1)


---

## 4 — DKT multi-run (10 seeds × 5 assignments)

`set_global_seed(seed)` chamado externamente antes de cada run (compensa que `dkt.py::train_dkt` não chama `torch.cuda.manual_seed_all` nem `cudnn.deterministic`). Prints em tempo real por run.

In [6]:
dkt_multirun = {}

print("=== DKT multi-run ===\n", flush=True)
t_dkt = time.time()

for aid in ASSIGNMENT_IDS:
    pidx = build_problem_index(seqs["train"][aid] + seqs["test"][aid])
    runs = []
    pred_df_seed42 = None
    n_train_events = n_test_events = None

    print(f"--- A{aid} (M={len(pidx)}) ---", flush=True)
    t_aid = time.time()
    for seed in SEEDS:
        set_global_seed(seed)
        if device.type == "cuda":
            torch.cuda.empty_cache()
        t_run = time.time()
        res = silent_train_and_evaluate_dkt(
            seqs["train"][aid], seqs["test"][aid],
            pidx, BEST_DKT_CONFIG, seed=seed,
        )
        elapsed = time.time() - t_run
        runs.append({
            "seed":      seed,
            "all_auc":   float(res["all_auc"]),
            "first_auc": float(res["first_auc"]),
            "pred_df":   None,
        })
        if seed == SEED_DEFAULT:
            # captura pred_df inline (sem refit)
            pred_df_seed42 = predict_dkt(
                res["model"], seqs["test"][aid], pidx,
                max_len=BEST_DKT_CONFIG["max_len"],
            )
            runs[-1]["pred_df"] = pred_df_seed42
        n_train_events = res["n_train_events"]
        n_test_events  = res["n_test_events"]
        print(f"  seed={seed}: all={fmt_pct(res['all_auc'])}, first={fmt_pct(res['first_auc'])} ({elapsed:.1f}s)", flush=True)
        del res
        if device.type == "cuda":
            torch.cuda.empty_cache()

    agg = aggregate_runs(runs)
    dkt_multirun[aid] = {
        **agg,
        "runs":           runs,
        "n_train_events": n_train_events,
        "n_test_events":  n_test_events,
        "config":         BEST_DKT_CONFIG,
    }
    print(f"  → A{aid} mean: first={fmt_pct(agg['first_auc_mean'])} ± {fmt_pct(agg['first_auc_std'])} "
          f"({(time.time()-t_aid)/60:.1f} min)\n", flush=True)

print(f"DKT multi-run total: {(time.time()-t_dkt)/60:.1f} min", flush=True)


=== DKT multi-run ===



--- A439 (M=10) ---


  seed=42: all=72.84%, first=78.77% (1.6s)


  seed=43: all=72.52%, first=77.35% (0.7s)


  seed=44: all=71.99%, first=77.41% (0.7s)


...

  seed=49: all=72.87%, first=80.75% (0.5s)


  seed=50: all=74.40%, first=84.09% (0.5s)


  seed=51: all=72.89%, first=81.23% (0.5s)


  → A502 mean: first=80.78% ± 4.01% (0.1 min)



DKT multi-run total: 0.6 min


---

## 5 — Code-DKT multi-run (10 seeds × 5 assignments)

Vocabulário construído por assignment uma única vez (depende só dos CodeStateIDs do train, não da seed).

In [7]:
def build_vocab_for_assignment(aid: int, _seqs, _cache):
    csids = set()
    for s in _seqs["train"][aid]:
        csids.update(s["events"]["CodeStateID"].astype(str).values)
    sub = {c: _cache[c] for c in csids if c in _cache}
    t2i, p2i = build_vocab(sub)
    return dict(token_to_idx=t2i, path_to_idx=p2i,
                node_count=len(t2i), path_count=len(p2i))


In [8]:
cdkt_multirun = {}

print("=== Code-DKT multi-run ===\n", flush=True)
t_cdkt = time.time()

for aid in ASSIGNMENT_IDS:
    vocab = build_vocab_for_assignment(aid, seqs, cache_raw)
    pidx  = build_problem_index(seqs["train"][aid] + seqs["test"][aid])
    runs = []
    n_train_events = n_test_events = None
    cdkt_seed42_state = None

    print(f"--- A{aid} (vocab: tok={vocab['node_count']}, path={vocab['path_count']}, M={len(pidx)}) ---", flush=True)
    t_aid = time.time()
    for seed in SEEDS:
        set_global_seed(seed)
        if device.type == "cuda":
            torch.cuda.empty_cache()
        t_run = time.time()
        res = silent_train_and_evaluate_cdkt(
            seqs["train"][aid], seqs["test"][aid],
            pidx, vocab, BEST_CDKT_CONFIG, cache_raw, seed=seed,
        )
        elapsed = time.time() - t_run
        runs.append({
            "seed":      seed,
            "all_auc":   float(res["all_auc"]),
            "first_auc": float(res["first_auc"]),
            "pred_df":   None,
        })
        if seed == SEED_DEFAULT:
            runs[-1]["pred_df"] = res["pred_df"]
            cdkt_seed42_state = {k: v.cpu() for k, v in res["model"].state_dict().items()}
        n_train_events = res["n_train_events"]
        n_test_events  = res["n_test_events"]
        print(f"  seed={seed}: all={fmt_pct(res['all_auc'])}, first={fmt_pct(res['first_auc'])} ({elapsed:.1f}s)", flush=True)
        del res
        if device.type == "cuda":
            torch.cuda.empty_cache()

    agg = aggregate_runs(runs)
    cdkt_multirun[aid] = {
        **agg,
        "runs":            runs,
        "n_train_events":  n_train_events,
        "n_test_events":   n_test_events,
        "config":          BEST_CDKT_CONFIG,
        "vocab":           vocab,
        "problem_to_idx":  pidx,
        "model_state_dict_seed42": cdkt_seed42_state,
    }
    print(f"  → A{aid} mean: first={fmt_pct(agg['first_auc_mean'])} ± {fmt_pct(agg['first_auc_std'])} "
          f"({(time.time()-t_aid)/60:.1f} min)\n", flush=True)

print(f"Code-DKT multi-run total: {(time.time()-t_cdkt)/60:.1f} min", flush=True)


=== Code-DKT multi-run ===



--- A439 (vocab: tok=494, path=21717, M=10) ---


  seed=42: all=69.23%, first=72.55% (11.6s)


  seed=43: all=69.89%, first=72.86% (11.7s)


  seed=44: all=69.57%, first=73.47% (11.6s)


...

  seed=49: all=75.21%, first=85.73% (9.3s)


  seed=50: all=76.89%, first=84.87% (9.2s)


  seed=51: all=76.36%, first=85.42% (9.3s)


  → A502 mean: first=84.98% ± 1.01% (1.5 min)



Code-DKT multi-run total: 8.7 min


---

## 6 — Sumário rápido (preview para 07_comparison)

In [9]:
rows = []
for aid in ASSIGNMENT_IDS:
    rows.append({
        "Assignment":     f"A{aid}",
        "BKT_first":      f"{fmt_pct(bkt_multirun[aid]['first_auc_mean'])} (n=1)",
        "DKT_first":      f"{fmt_pct(dkt_multirun[aid]['first_auc_mean'])} ± {fmt_pct(dkt_multirun[aid]['first_auc_std'])}",
        "CodeDKT_first":  f"{fmt_pct(cdkt_multirun[aid]['first_auc_mean'])} ± {fmt_pct(cdkt_multirun[aid]['first_auc_std'])}",
        "BKT_all":        f"{fmt_pct(bkt_multirun[aid]['all_auc_mean'])} (n=1)",
        "DKT_all":        f"{fmt_pct(dkt_multirun[aid]['all_auc_mean'])} ± {fmt_pct(dkt_multirun[aid]['all_auc_std'])}",
        "CodeDKT_all":    f"{fmt_pct(cdkt_multirun[aid]['all_auc_mean'])} ± {fmt_pct(cdkt_multirun[aid]['all_auc_std'])}",
    })
summary_df = pd.DataFrame(rows)
print("=== Sumário multi-run (BKT=1 run, DKT/CDKT=10 runs) ===\n")
print(summary_df.to_string(index=False))


=== Sumário multi-run (BKT=1 run, DKT/CDKT=10 runs) ===

Assignment    BKT_first      DKT_first  CodeDKT_first      BKT_all        DKT_all    CodeDKT_all
      A439 63.21% (n=1) 75.56% ± 3.40% 73.27% ± 1.34% 64.23% (n=1) 70.89% ± 2.15% 70.35% ± 0.67%
      A487 68.40% (n=1) 76.70% ± 3.24% 79.56% ± 0.76% 69.07% (n=1) 72.77% ± 1.18% 74.89% ± 0.61%
      A492 54.20% (n=1) 82.05% ± 1.66% 86.12% ± 0.53% 63.62% (n=1) 76.66% ± 0.84% 79.08% ± 0.74%
      A494 57.81% (n=1) 80.17% ± 5.04% 81.85% ± 0.98% 59.66% (n=1) 72.57% ± 2.79% 75.07% ± 0.97%
      A502 56.92% (n=1) 80.78% ± 4.01% 84.98% ± 1.01% 57.37% (n=1) 72.48% ± 1.88% 76.24% ± 0.77%


---

## 7 — Serialização

In [10]:
paths_to_save = [
    (RESULTS_DIR / "bkt_results_multirun.pkl",      bkt_multirun),
    (RESULTS_DIR / "dkt_results_multirun.pkl",      dkt_multirun),
    (RESULTS_DIR / "code_dkt_results_multirun.pkl", cdkt_multirun),
]
for path, obj in paths_to_save:
    with open(path, "wb") as f:
        pickle.dump(obj, f, protocol=4)
    print(f"Salvo: {path.name:<37}  {path.stat().st_size/1e6:>7.1f} MB")


Salvo: bkt_results_multirun.pkl                   0.0 MB
Salvo: dkt_results_multirun.pkl                   0.3 MB
Salvo: code_dkt_results_multirun.pkl             79.9 MB


---

## 8 — Sanity checks

In [11]:
# Schema check
for model_name, obj, expected_runs in [
    ("bkt", bkt_multirun, 1),
    ("dkt", dkt_multirun, 10),
    ("code_dkt", cdkt_multirun, 10),
]:
    for aid in ASSIGNMENT_IDS:
        runs = obj[aid]["runs"]
        assert len(runs) == expected_runs, f"{model_name} A{aid}: {len(runs)} runs (esperado {expected_runs})"
        seeds_found = sorted([r["seed"] for r in runs])
        expected_seeds = [SEED_DEFAULT] if expected_runs == 1 else list(SEEDS)
        assert seeds_found == expected_seeds, f"{model_name} A{aid}: seeds={seeds_found}"

print("Schema check: ✓ BKT(1 run) + DKT(10 runs) + Code-DKT(10 runs) × 5 assignments")


Schema check: ✓ BKT(1 run) + DKT(10 runs) + Code-DKT(10 runs) × 5 assignments


In [12]:
# Coerência seed=42 multirun vs single-run
TOL = 0.03  # 3pp de tolerância (CUDA não-determinismo em LSTM)

def cmp_pair(name, single, multi, aid):
    single_first = single[aid]["first_auc"]
    multi_seed42 = next(r["first_auc"] for r in multi[aid]["runs"] if r["seed"] == 42)
    diff = abs(single_first - multi_seed42) if single_first is not None else float("nan")
    ok = "✓" if (single_first is None or diff <= TOL) else "⚠"
    sf = f"{single_first:.4f}" if single_first is not None else "N/A"
    print(f"  {ok} {name} A{aid}: single={sf} vs multirun_seed42={multi_seed42:.4f} (Δ={diff:.4f})")
    return single_first is None or diff <= TOL

print("--- BKT (1 run, deve ser idêntico) ---")
all_ok = True
for aid in ASSIGNMENT_IDS: all_ok &= cmp_pair("BKT",     bkt_single,  bkt_multirun,  aid)
print("--- DKT ---")
for aid in ASSIGNMENT_IDS: all_ok &= cmp_pair("DKT",     dkt_single,  dkt_multirun,  aid)
print("--- Code-DKT ---")
for aid in ASSIGNMENT_IDS: all_ok &= cmp_pair("CodeDKT", cdkt_single, cdkt_multirun, aid)
print()
print("✓ Coerência OK." if all_ok else "⚠ Investigar divergências.")


--- BKT (1 run, deve ser idêntico) ---
  ✓ BKT A439: single=0.6321 vs multirun_seed42=0.6321 (Δ=0.0000)
  ✓ BKT A487: single=0.6840 vs multirun_seed42=0.6840 (Δ=0.0000)
  ✓ BKT A492: single=0.5420 vs multirun_seed42=0.5420 (Δ=0.0000)
  ✓ BKT A494: single=0.5781 vs multirun_seed42=0.5781 (Δ=0.0000)
  ✓ BKT A502: single=0.5692 vs multirun_seed42=0.5692 (Δ=0.0000)
--- DKT ---
  ✓ DKT A439: single=0.7877 vs multirun_seed42=0.7877 (Δ=0.0000)
  ✓ DKT A487: single=0.7593 vs multirun_seed42=0.7593 (Δ=0.0000)
  ✓ DKT A492: single=0.8292 vs multirun_seed42=0.8292 (Δ=0.0000)
  ✓ DKT A494: single=0.7923 vs multirun_seed42=0.7923 (Δ=0.0000)
  ✓ DKT A502: single=0.8435 vs multirun_seed42=0.8435 (Δ=0.0000)
--- Code-DKT ---
  ✓ CodeDKT A439: single=0.7255 vs multirun_seed42=0.7255 (Δ=0.0000)
  ✓ CodeDKT A487: single=0.7918 vs multirun_seed42=0.7918 (Δ=0.0000)
  ✓ CodeDKT A492: single=0.8617 vs multirun_seed42=0.8617 (Δ=0.0000)
  ✓ CodeDKT A494: single=0.8296 vs multirun_seed42=0.8296 (Δ=0.0000)
  ✓ Co